In [22]:
import os 
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import sys
sys.path.append('./models/')
from useful_functions import df_to_dict, concat_dico, get_classement, sort_list


In [23]:
models1Dnames=['Moving Average','ARIMA', 'Exponential', 'Linear Regression', 'Bayesian Regression','SIRH1', 'SIRH2', 'SIRH3', 'SIRH4']
models3Dnames=[ 'VAR', 'Exponential Multi', 'SIRH Multi1', 'SIRH Multi2','SEIR Mob']
list_of_models= models1Dnames+models3Dnames
print(list_of_models)

['Moving Average', 'ARIMA', 'Exponential', 'Linear Regression', 'Bayesian Regression', 'SIRH1', 'SIRH2', 'SIRH3', 'SIRH4', 'VAR', 'Exponential Multi', 'SIRH Multi1', 'SIRH Multi2', 'SEIR Mob']


In [24]:
def classify_bis(point, r_effs):  # Classification based on transmission dynamics

    if r_effs[point] < 0.5:
        return 'minimal transmission'
    elif r_effs[point] < 0.8:
        return 'low transmission'
    elif r_effs[point] < 1.2:
        return 'stable'
    elif r_effs[point] < 3:
        return 'high transmission'
    else:
        return 'very high transmission'

In [25]:
def delete_rand_items(items,n):
    to_delete = set(random.sample(range(len(items)),n))
    return [x for i,x in enumerate(items) if not i in to_delete]

In [26]:
def process_row(row):
    smallest_indices = row.nsmallest(5).index  # Get the 5 smallest value indices
    row_transformed = pd.Series(0, index=row.index)  # Initialize row with zeros
    row_transformed[smallest_indices] = 1 / row[smallest_indices]  # Invert smallest values
    total_sum = row_transformed.sum()
    if total_sum > 0:
        row_transformed /= total_sum
    return row_transformed

In [45]:
def get_weights(n):
    model_type='1D'
    loss='WIS'
    reach='14'
    list_of_models= models1Dnames+models3Dnames
    type_of_points=['all','very high transmission', 'high transmission' , 'stable',  'low transmission', 'minimal transmission']
    items=[name for name in os.listdir('./results/global_evaluation/') if loss in name and '1D' in name and 'reach_='+str(reach) in name   ] # results of the models
    results_list=delete_rand_items(items,n)            
    for loss in ['RMSE']: #'WIS' 
        for reach in ['14']: #'14'
            df_expected_ranks=pd.DataFrame(columns= list_of_models, index = type_of_points )
    
            for label_point in type_of_points:
    
                all_ranks=np.zeros((len(list_of_models), len(list_of_models)))
                for name in results_list :
                    mob=int(name.split('_')[-5])
                    pand=int(name.split('_')[-4])
                    with open('./results/global_evaluation/'+name, 'r') as f:
                        dicoresults1 = json.load(f)
                    with open('./results/global_evaluation/'+name.replace('1D', '3D'), 'r') as f:
                        dicoresults2 = json.load(f)
                    dicoresults=concat_dico(dicoresults1, dicoresults2)
                    df=pd.read_csv('./all_pandemics/pandemic_'+name.split('_')[-5]+'_'+name.split('_')[-4]+'.csv')
                    df.index=['n_hospitalized', 'n_infectious', 'mobility', 'r_eff']
                    df.drop(['Unnamed: 0'], axis=1, inplace=True)
                    n_hospitalized = np.array(df.loc['n_hospitalized'])
                    r_eff=np.array(df.loc['r_eff'])
                    indexs_points=[[20*i] for i in range(1, 15) ] 
                    prediction=pd.read_csv('./results/predictions_of_the_models/predictions_'+str(reach)+'_days_on_pandemic_'+str(mob)+'_'+str(pand)+'.csv')
                    prediction.drop(['Unnamed: 0'], axis=1, inplace=True)
                    prediction.index=[20*i for i in range(1, 15)]
                    #prediction_bis=prediction.drop(['Real values', 'Exponential', 'Exponential Multi'], axis=1)
                    prediction_all=prediction
                    #prediction=prediction_bis
                    for point in indexs_points: 
                        if n_hospitalized[point[0]] >= 10:
                            if label_point=='all': 
                                real_value=prediction_all['Real values'].loc[point[0]]
                                perfs=dicoresults[str(point)]
                                #assert(len(perfs)==14)
                                #assert(abs(perfs[2] - perfs[11]) < 0.001, (perfs[2], perfs[11])) 
                                #perfs.pop(11) # remove the moving average multi
                                #assert(len(perfs)==13)
                                rankings=get_classement(perfs)
                                for i in range(len(rankings)): 
                                    rank_model_i=rankings[i]
                                    all_ranks[i][rank_model_i]+=1
    
                            elif classify_bis(point[0], r_eff) == label_point :
                                real_value=prediction_all['Real values'].loc[point[0]]
                                perfs=dicoresults[str(point)]
                                #assert(len(perfs)==14)
                                #assert(abs(perfs[2] - perfs[11]) < 0.001, (perfs[2], perfs[11])) 
                                #perfs.pop(11) # remove the moving average multi
                                #assert(len(perfs)==13)
                                rankings=get_classement(perfs)
                                for i in range(len(rankings)): 
                                    rank_model_i=rankings[i]
                                    all_ranks[i][rank_model_i]+=1
                expected_ranks=[np.sum((np.array(all_ranks[i])/np.sum(np.array(all_ranks[i])))*np.array([i for i in range( len(rankings))])) for i in range(len(rankings))]    
                df_expected_ranks.loc[label_point]=expected_ranks
    df_ranks=df_expected_ranks.copy()
    df_ranks = df_ranks.apply(pd.to_numeric, errors='coerce')
    df_ranks = df_ranks.apply(process_row, axis=1)
    return df_ranks
    
                            

In [46]:
N=[150,175,200,225,250,275,300] #antalet pandemier som plockas bort
reps=10
for n in N:
    for i in range(reps):
        w=get_weights(n)
        w.to_csv('./ranks_for_ensemble_test'+str(n)+'_'+str(i)+'.csv')

/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_43242/787612692.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.24347027 0.20579099 0.19511939 0.19347101 0.16572858]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  row_transformed[smallest_indices] = 1 / row[smallest_indices]  # Invert smallest values
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_43242/787612692.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.56595745 0.31367925 0.22428331 0.20493066 0.19444444]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  row_transformed[smallest_indices] = 1 / row[smallest_indices]  # Invert smallest values
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_43242/787612692.py:4: FutureWarning: Setting an item of incompatible dtyp

In [44]:
w=get_weights(300)
w.to_csv('./ranks_for_ensemble_test'+'300'+'.csv')

/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_43242/787612692.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.28407789 0.21792619 0.19792498 0.1898928  0.16272966]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  row_transformed[smallest_indices] = 1 / row[smallest_indices]  # Invert smallest values
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_43242/787612692.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.8125     0.26530612 0.26       0.18055556 0.17808219]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  row_transformed[smallest_indices] = 1 / row[smallest_indices]  # Invert smallest values
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_43242/787612692.py:4: FutureWarning: Setting an item of incompatible dtyp

14
